# Pixel-resolution phase map — Hann-weighted K-means clustering

Third of three notebooks split from `correlogram_pca_full.ipynb`. Fits K-means on the 10 PCA components alone (no `tile_var` — unlike `tile_phase_map.ipynb`), then blends each tile's PCA score onto pixel space with the same Hann window used for that tile's own FFT (each pixel's vote weighted by how much it actually contributed to its covering tiles' signatures), and classifies each pixel against the fitted centroids. This gives smooth, contiguous cluster boundaries in place of the blocky tile grid. Clusters are left **numbered** (`Cluster 0..N-1`), not semantically labeled — see `tile_phase_map.ipynb` for the labeled void/amorphous/monolayer/bilayer version.

**Self-contained** — runs its own `run_dynamic_pipeline()` call (batched GPU correlogram generation + PCA + K-means, `twist_mapper_GPU/dynamic_pipeline.py`), decoupled from `correlogram_pca_full.ipynb`'s on-disk cache. If a `dynamic_result` from a prior run already exists in this kernel session, that's reused directly instead of recomputing. The K-means fit *inside* `run_dynamic_pipeline` is already exactly the PC-only clustering this notebook wants, so it's reused directly too — no separate K-means cell needed here (contrast `tile_phase_map.ipynb`, which fits its own K-means on a different feature space: PCA + tile variance).

**Data:** runs on the sample image `data/moire.npz` by default — fetch it first with `python download_sample_data.py` (from this `examples/` folder; see `data/README.md` for the source and citation). To analyze your own data, set `SOURCE_PATH` below to any video, image, or `.npz` file.

(`correlogram_pca_full.ipynb` and `tile_phase_map.ipynb`, referenced above, are development notebooks not included in this repository.)

## Setup

Loads the video (or image/.npz -- `load_video_tensor` handles any of them) and runs the dynamic pipeline: tiling -> batched correlogram generation -> PCA -> K-means.

In [ ]:
import os
import time
import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

from twist_mapper_GPU import (TileFFTAnalysis, load_video_tensor, build_symmetric_hann2d, blend_and_classify_frame,
                              extract_pixel_size_ang, default_d_spacing_ang, get_material)
from twist_mapper_GPU.dynamic_pipeline import run_dynamic_pipeline

print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
SOURCE_PATH = os.path.join('data', 'moire.npz')  # or your own video / image / .npz
TILE_SIZE = 128
STRIDE = 32
PIXEL_SIZE_ANG = extract_pixel_size_ang(SOURCE_PATH) or 0.25  # from file metadata (.npz), else manual
MATERIAL = 'WSSe'
D_SPACING_ANG = default_d_spacing_ang(get_material(MATERIAL))
SUBTRACT_MEAN = False  # TileFFTAnalysis/BatchedCorrelogramPipeline: subtract each tile's
                       # mean before windowing/FFT (reduces DC; does not sharpen
                       # Bragg peaks — see TileFFTAnalysis._compute_fft_magnitude)
APPLY_CLAHE = False    # global, per-frame CLAHE before tiling — see
                       # twist_mapper_GPU.video_preprocessing.apply_clahe_to_video
N_COMPONENTS = 10
N_PC_CLUSTERS = 5

device = 'cuda' if torch.cuda.is_available() else 'cpu'

CACHE_DIR = 'cache'  # per-frame area fractions are cached here (examples/cache/, git-ignored)
os.makedirs(CACHE_DIR, exist_ok=True)
# Cache key includes the source file name, so switching SOURCE_PATH never reuses stale results
_suffix = ('_' + os.path.splitext(os.path.basename(SOURCE_PATH))[0]
           + f'_t{TILE_SIZE}_s{STRIDE}'
           + ('_meansub' if SUBTRACT_MEAN else '') + ('_clahe' if APPLY_CLAHE else ''))

# Reuse an in-memory result from a prior run in this same kernel session
# (e.g. if correlogram_pca_full.ipynb's or tile_phase_map.ipynb's cells
# were already run here), else compute fresh — the whole pipeline now
# takes seconds, not minutes, so there's no need to depend on a cache
# file written by a separate notebook.
if 'dynamic_result' not in globals():
    video_tensor = load_video_tensor(SOURCE_PATH, device)
    expected_radius = TileFFTAnalysis.expected_ring_radius(TILE_SIZE, PIXEL_SIZE_ANG, D_SPACING_ANG)
    dynamic_result = run_dynamic_pipeline(
        video_tensor, tile_size=TILE_SIZE, stride=STRIDE, expected_radius=expected_radius,
        n_components=N_COMPONENTS, n_clusters=N_PC_CLUSTERS, apply_clahe=APPLY_CLAHE,
        crop_black_margins=True, subtract_mean=SUBTRACT_MEAN, verbose=True,
    )
    print(f"Dynamic pipeline timings: {dynamic_result.timings}")
else:
    print("Reusing in-memory dynamic_result from this session")

tiled = dynamic_result.tiled
Frames, Rows, Cols = dynamic_result.Frames, dynamic_result.Rows, dynamic_result.Cols
H, W = tiled.video.shape[1:]

# run_dynamic_pipeline's own internal K-means IS exactly the PC-only fit
# this notebook wants (n_clusters=N_PC_CLUSTERS above) — reuse its
# labels/centroids directly rather than fitting a second, redundant
# K-means (contrast tile_phase_map.ipynb, which needs its own separate
# fit on a different feature space: PCA + tile variance).
pc_cluster_labels = dynamic_result.cluster_labels
pc_centroids = dynamic_result.centroids
explained = dynamic_result.explained_variance_ratio

print(f"Frames={Frames}, Rows={Rows}, Cols={Cols}, total tiles={Frames * Rows * Cols}")
print(f"pc_centroids shape: {pc_centroids.shape}  (N_PC_CLUSTERS x N_COMPONENTS)")
print(f"\n{'Cluster':<10}{'Tiles':<12}{'Fraction':<10}")
for k in range(N_PC_CLUSTERS):
    mask = pc_cluster_labels == k
    print(f"{k:<10}{mask.sum():<12}{mask.mean() * 100:<9.1f}%")

In [ ]:
# Standardize PCA scores (mirrors run_dynamic_pipeline's own internal
# standardization exactly -- pc_centroids were fit in this same
# standardized space, so the pixel-blend field must use it too, not raw
# X_pca: PCA orders components by decreasing variance, so raw PC1 has
# much larger magnitude than raw PC8, and nearest-centroid classification
# against raw-scale blended vectors would compare mismatched scales).
X_pca_np = dynamic_result.X_pca.detach().cpu().numpy()
X_pca_z = (X_pca_np - X_pca_np.mean(axis=0)) / (X_pca_np.std(axis=0) + 1e-8)

X_pca_z_torch = torch.as_tensor(X_pca_z, device=tiled.video.device, dtype=torch.float32)
X_pca_grid = X_pca_z_torch.reshape(Frames, Rows, Cols, N_COMPONENTS)
centroids_torch = torch.as_tensor(pc_centroids, device=tiled.video.device, dtype=torch.float32)
hann2d = build_symmetric_hann2d(tiled.tile_size, device=tiled.video.device)

qual_cmap = matplotlib.colormaps['tab10'].resampled(N_PC_CLUSTERS)
pixel_cmap = ListedColormap([qual_cmap(i) for i in range(N_PC_CLUSTERS)])

## Pixel-resolution cluster map: Hann-weighted PC blending

The phase maps above are at *tile* resolution (one label per tile) — each tile
gets one label, plotted as one cell, which doesn't reflect that tiles
overlap 75% (`tile_size=128`, `stride=32`: every interior pixel sits inside
up to `(128/32)^2 = 16` different tiles) or that each tile's own correlogram
was itself built with a Hann-windowed FFT, so pixels near a tile's edge
barely influenced that tile's score in the first place.

This section assigns a cluster to every *pixel* instead, by reusing the
same Hann window as the vote weight: for each pixel, blend the PC scores of
every tile covering it, weighted by that pixel's own Hann value within each
tile (exactly how much that pixel contributed to that tile's signature) —
see `twist_mapper_GPU/pixel_cluster_map.py`. Because PCA is linear, blending
the already-computed 10-D `X_pca` scores this way is mathematically
identical to Hann-blending the raw 360-D correlograms first and then
projecting, just far cheaper.

Two things different from the phase-clustering section above, deliberately:
- **Clustering features are just the 10 PCs** (no `tile_var`) — a separate,
  simpler K-means fit from `kmeans`/`cluster_labels` above.
- **No semantic labeling.** Clusters are left as plain `Cluster 0`..`Cluster
  {N-1}` — the void/amorphous/monolayer/bilayer guessing heuristic isn't
  reapplied here.

The fitted centroids are cached to disk so `cluster_map_inspector_app.py`
(a separate live Panel/Bokeh app) can classify pixels the same way without
needing cuML at inference time — just nearest-centroid distance.

In [ ]:
FRAMES_TO_SHOW_PIXEL = sorted({0, Frames // 2, Frames - 1})  # just [0] for a single image

fig, axes = plt.subplots(1, len(FRAMES_TO_SHOW_PIXEL), figsize=(4.8 * len(FRAMES_TO_SHOW_PIXEL), 4.8),
                         squeeze=False)
axes = axes[0]
for ax, f in zip(axes, FRAMES_TO_SHOW_PIXEL):
    t0 = time.time()
    pixel_labels, covered = blend_and_classify_frame(
        X_pca_grid[f], centroids_torch, tiled.stride, H, W, hann2d
    )
    elapsed = time.time() - t0

    display = pixel_labels.float().detach().cpu().numpy()
    display[~covered.detach().cpu().numpy()] = np.nan  # masked -> shows as blank

    im = ax.imshow(display, cmap=pixel_cmap, vmin=-0.5, vmax=N_PC_CLUSTERS - 0.5, origin='upper')
    ax.set_title(f'Frame {f} ({elapsed * 1000:.0f} ms)', fontsize=10, weight='bold')
    ax.set_xlabel('Pixel col')
    if ax is axes[0]:
        ax.set_ylabel('Pixel row')

legend_handles = [Patch(facecolor=qual_cmap(i), label=f'Cluster {i}') for i in range(N_PC_CLUSTERS)]
fig.legend(handles=legend_handles, loc='lower center', ncol=N_PC_CLUSTERS, bbox_to_anchor=(0.5, -0.05))
fig.suptitle('Pixel-resolution cluster map (Hann-weighted PC blend + nearest centroid)',
            fontsize=13, weight='bold')
plt.tight_layout()
plt.show()

### Continuity check: raw PCA score map (single component, unclassified)

The cluster map above is a *hard* nearest-centroid label — a tile/pixel near a decision
boundary can flip labels frame-to-frame from ordinary correlogram noise, even though the
underlying PCA score is moving smoothly. To check whether the representation itself
(before K-means discretizes it) varies continuously in space and time, plot the raw
Hann-blended score of a single PCA component (`blend_pc_field`, no `classify_nearest_centroid`
step) with a continuous colormap. Domain boundaries growing/shrinking should show up as smooth
gradients here even if the discrete cluster map above flickers.

Set `PC_COMPONENT_TO_PLOT` to inspect a different component (0-indexed, `< N_COMPONENTS`).

In [ ]:
from twist_mapper_GPU import blend_pc_field

PC_COMPONENT_TO_PLOT = 0  # which of the N_COMPONENTS PCs to visualize (0-indexed)

fig, axes = plt.subplots(1, len(FRAMES_TO_SHOW_PIXEL), figsize=(4.8 * len(FRAMES_TO_SHOW_PIXEL), 4.8),
                         squeeze=False)
axes = axes[0]
score_maps = []
for f in FRAMES_TO_SHOW_PIXEL:
    pc_field, covered = blend_pc_field(X_pca_grid[f], tiled.stride, H, W, hann2d)
    score = pc_field[..., PC_COMPONENT_TO_PLOT].detach().cpu().numpy()
    score[~covered.detach().cpu().numpy()] = np.nan
    score_maps.append(score)

vmax = np.nanmax(np.abs(score_maps))
for ax, f, score in zip(axes, FRAMES_TO_SHOW_PIXEL, score_maps):
    im = ax.imshow(score, cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='upper')
    ax.set_title(f'Frame {f}', fontsize=10, weight='bold')
    ax.set_xlabel('Pixel col')
    if ax is axes[0]:
        ax.set_ylabel('Pixel row')

fig.colorbar(im, ax=axes, shrink=0.8, label=f'PC{PC_COMPONENT_TO_PLOT} score (standardized)')
fig.suptitle(f'Pixel-resolution PC{PC_COMPONENT_TO_PLOT} score map (Hann-weighted blend, unclassified)',
            fontsize=13, weight='bold')
plt.show()

### Interactive: scrub the PC score map frame-by-frame

Same raw (unclassified) Hann-blended PC score map as above, but with a frame slider and a
component dropdown so you can scrub through the whole video like a movie and watch whether
domain boundaries drift continuously or jump. The figure is built once and only its image
data/title are mutated per frame, rendered to a persistent `ipywidgets.Image` -- no
clear-and-rebuild of the plot, so dragging the slider updates smoothly instead of flashing/
refreshing the whole output on every tick. Works with any matplotlib backend (no
`ipympl`/`%matplotlib widget` needed).

In [ ]:
import io
import ipywidgets as widgets
from IPython.display import display

# Fixed color scale across all frames (computed once from a handful of sampled frames)
# so brightness changes reflect the score itself, not an autoscaled colorbar.
_sample_frames = np.linspace(0, Frames - 1, min(20, Frames), dtype=int)
_sample_vmax = 0.0
for f in _sample_frames:
    pc_field, covered = blend_pc_field(X_pca_grid[int(f)], tiled.stride, H, W, hann2d)
    score = pc_field[..., PC_COMPONENT_TO_PLOT].detach().cpu().numpy()
    _sample_vmax = max(_sample_vmax, np.nanmax(np.abs(score[covered.detach().cpu().numpy()])))

# Built ONCE and mutated in place (im_i.set_data / set_text) rather than recreated on every
# slider tick -- avoids the flicker of tearing down and rebuilding the whole figure/axes/
# colorbar each frame. Never displayed via pyplot (closed immediately); it only ever gets
# rendered off-screen to a PNG buffer pushed into image_widget below, so the DOM element
# being updated is a single <img>, not a whole new plot each time.
fig_i, ax_i = plt.subplots(figsize=(5.5, 5.5))
plt.close(fig_i)
im_i = ax_i.imshow(np.full((H, W), np.nan), cmap='RdBu_r', vmin=-_sample_vmax, vmax=_sample_vmax,
                   origin='upper')
cbar_i = fig_i.colorbar(im_i, ax=ax_i, shrink=0.85)
title_i = ax_i.set_title('Frame 0')
ax_i.set_xlabel('Pixel col')
ax_i.set_ylabel('Pixel row')

frame_slider_nb = widgets.IntSlider(
    min=0, max=Frames - 1, step=1, value=0, description='Frame',
    continuous_update=True, layout=widgets.Layout(width='700px'),
)
component_dd_nb = widgets.Dropdown(options=list(range(N_COMPONENTS)), value=PC_COMPONENT_TO_PLOT,
                                   description='PC component')
play_nb = widgets.Play(min=0, max=Frames - 1, step=1, interval=1000, description='Press play')
widgets.jslink((play_nb, 'value'), (frame_slider_nb, 'value'))

image_widget = widgets.Image(format='png', width=550, height=550)
display(widgets.VBox([image_widget, widgets.HBox([play_nb, frame_slider_nb, component_dd_nb])]))


def _update_score_map(frame, component):
    pc_field, covered = blend_pc_field(X_pca_grid[frame], tiled.stride, H, W, hann2d)
    covered_np = covered.detach().cpu().numpy()
    score = pc_field[..., component].detach().cpu().numpy()
    score[~covered_np] = np.nan

    im_i.set_data(score)
    cbar_i.set_label(f'PC{component} score (standardized)')
    title_i.set_text(f'Frame {frame}')

    buf = io.BytesIO()
    fig_i.savefig(buf, format='png', bbox_inches='tight')
    image_widget.value = buf.getvalue()
    buf.close()


widgets.interactive_output(_update_score_map, {'frame': frame_slider_nb, 'component': component_dd_nb})

### Dynamics: cluster area-fraction vs frame

Same idea as `tile_phase_map.ipynb`'s dynamics section, but at pixel resolution: for every frame, blend + classify every pixel, then report what fraction of the *covered* pixels (excludes the Hann-weight-zero border margin — see `blend_pc_field`'s docstring) each cluster occupies. Cached to disk since it's a full pass over all frames (a few seconds at ~20-60ms/frame, per the timings printed in the cluster-map plot above).

In [ ]:
CACHE_PIXEL_AREA_FRACTION = os.path.join(
    CACHE_DIR, f'pixel_phase_area_fraction_k{N_PC_CLUSTERS}{_suffix}.npy'
)

if os.path.exists(CACHE_PIXEL_AREA_FRACTION):
    print(f"Loading cached pixel-resolution area fractions from {CACHE_PIXEL_AREA_FRACTION} ...")
    pixel_phase_fraction = np.load(CACHE_PIXEL_AREA_FRACTION)
else:
    t0 = time.time()
    pixel_phase_fraction = np.zeros((Frames, N_PC_CLUSTERS))
    for f in range(Frames):
        labels, covered = blend_and_classify_frame(
            X_pca_grid[f], centroids_torch, tiled.stride, H, W, hann2d
        )
        labels_np = labels.detach().cpu().numpy()
        covered_np = covered.detach().cpu().numpy()
        n_covered = covered_np.sum()
        for k in range(N_PC_CLUSTERS):
            pixel_phase_fraction[f, k] = ((labels_np == k) & covered_np).sum() / n_covered
        if (f + 1) % 100 == 0 or f == Frames - 1:
            elapsed = time.time() - t0
            print(f"  {f + 1}/{Frames} frames ({elapsed:.0f}s elapsed)")
    print(f"Computed pixel-resolution area fractions for all {Frames} frames "
          f"in {time.time() - t0:.1f}s")

    np.save(CACHE_PIXEL_AREA_FRACTION, pixel_phase_fraction)
    print(f"Cached to {CACHE_PIXEL_AREA_FRACTION} for reuse")


In [ ]:
plt.figure(figsize=(11, 5))
for k in range(N_PC_CLUSTERS):
    plt.plot(np.arange(Frames), pixel_phase_fraction[:, k] * 100,
             label=f'Cluster {k}', color=qual_cmap(k), lw=1.6,
             marker='o' if Frames <= 20 else None)  # markers so a 1-frame source still shows
plt.xlabel('Frame')
plt.ylabel('Area fraction of covered pixels (%)')
plt.title('Pixel-resolution cluster area-fraction vs frame')
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
